<div align="center">

# 📝 NLP Foundations for ML Researchers

### Text Preprocessing, Vocabulary Engineering & Vectorization from Scratch

<a href="https://colab.research.google.com/github/mahfuzswe/python-for-ml-researchers/blob/main/NLP_for_ML_Researchers.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open in Colab" height="28"></a>

<img src="https://img.shields.io/badge/Python-3.10%2B-3776AB?style=flat-square&logo=python&logoColor=white" alt="Python 3.10+">
<img src="https://img.shields.io/badge/chapters-4-FFD43B?style=flat-square" alt="4 chapters">
<img src="https://img.shields.io/badge/time-45%20min-3FB950?style=flat-square" alt="45 min">
<img src="https://img.shields.io/badge/PRs-welcome-3FB950?style=flat-square" alt="PRs welcome">

</div>

---

## 💡 Why this exists

Language models, multimodal networks, and text classifiers all operate on token sequences and numerical embeddings. This notebook covers the fundamental transition from raw text strings to neural-ready numerical matrices in pure Python and NumPy.

---
# Part F · NLP and Text Processing for ML Researchers

## Chapter 27 · Text Tokenization and Normalization ⏱️ ~8 min

Before language models can process raw text, sentences must be broken down into discrete numerical units called **tokens**. 

In modern ML research, tokenization lies at the boundary between raw data and tensor operations:
* **Character-level**: No unknown words, but sequences become excessively long.
* **Word-level**: Simple and intuitive, but struggles with out-of-vocabulary (OOV) terms.
* **Sub-word (BPE / WordPiece)**: The industry standard in modern LLMs (LLaMA, GPT, BERT).

Before using complex tokenizers, researchers must understand **text normalization** (casing, punctuation stripping, contractions) and how to write clean, deterministic tokenizers in pure Python.

In [1]:
import re

# Sample research log authored by male research leads
research_logs = (
    "Tariq and Khalid trained the 7B parameter transformer on 8 A100 GPUs. "
    "Rahim observed a gradient anomaly at step 4200! David applied gradient clipping (max_norm=1.0), "
    "and training loss dropped smoothly from 3.84 to 1.92. Isn't that impressive?"
)

def tokenize_research_text(text: str) -> list[str]:
    """Standardizes text to lowercase and extracts word tokens and contraction words."""
    text = text.lower()
    # Matches words, numbers, and contractions like isn't
    tokens = re.findall(r"\b[a-z0-9]+(?:'[a-z]+)?\b", text)
    return tokens

tokens = tokenize_research_text(research_logs)
print(f"Total tokens extracted: {len(tokens)}")
print("First 10 tokens:", tokens[:10])

# Frequency distribution using pure Python dictionary
token_freqs = {}
for tok in tokens:
    token_freqs[tok] = token_freqs.get(tok, 0) + 1

# Top 5 most frequent tokens
sorted_freqs = sorted(token_freqs.items(), key=lambda item: -item[1])[:5]
print("Top 5 frequent tokens:", sorted_freqs)

Total tokens extracted: 40
First 10 tokens: ['tariq', 'and', 'khalid', 'trained', 'the', '7b', 'parameter', 'transformer', 'on', '8']
Top 5 frequent tokens: [('and', 2), ('gradient', 2), ('1', 2), ('tariq', 1), ('khalid', 1)]


### 🧠 Practice

1. Write a function `extract_researcher_mentions(text, known_names)` that returns all names from `known_names = ["khalid", "tariq", "rahim", "david"]` mentioned in a log string.
2. Given a list of sentences, write a list comprehension that tokenizes each sentence and returns only tokens with length >= 4.

<details>
<summary>💡 <b>Show solution</b> (try first!)</summary>

```python
known_names = {"khalid", "tariq", "rahim", "david"}
def extract_researcher_mentions(text, names_set):
    tokens = set(tokenize_research_text(text))
    return tokens.intersection(names_set)

print("Found researchers:", extract_researcher_mentions(research_logs, known_names))

sentences = ["Khalid benchmarks latency", "Tariq fine-tunes weights"]
long_tokens = [[tok for tok in tokenize_research_text(s) if len(tok) >= 4] for s in sentences]
print("Long tokens:", long_tokens)
```

</details>

## Chapter 28 · Vocabulary and Token-to-ID Indexing ⏱️ ~10 min

Neural networks cannot perform dot products on string characters. Every unique token in your dataset must map to a unique integer ID (`token_to_idx`), and back (`idx_to_token`).

PyTorch `nn.Embedding(num_embeddings, embedding_dim)` takes a tensor of these integer IDs as input.

### Special Tokens in ML Research
* `<pad>` (index 0): Pads variable-length sentences in a batch to identical length.
* `<unk>` (index 1): Replaces words unseen during training to prevent crashes.
* `<bos>` / `<eos>`: Marks the Beginning and End of Sequence for autoregressive generation.

In [2]:
class Vocabulary:
    """Production-ready vocabulary mapping class built in pure Python."""
    def __init__(self, pad_token="<pad>", unk_token="<unk>", min_freq=1):
        self.pad_token = pad_token
        self.unk_token = unk_token
        self.min_freq = min_freq
        
        self.token_to_idx = {self.pad_token: 0, self.unk_token: 1}
        self.idx_to_token = {0: self.pad_token, 1: self.unk_token}
        self.counts = {}

    def build_vocab(self, corpus: list[str]):
        """Extract unique words and assign index IDs ordered by frequency."""
        for doc in corpus:
            for tok in tokenize_research_text(doc):
                self.counts[tok] = self.counts.get(tok, 0) + 1
                
        for tok, count in sorted(self.counts.items(), key=lambda item: -item[1]):
            if count >= self.min_freq and tok not in self.token_to_idx:
                new_idx = len(self.token_to_idx)
                self.token_to_idx[tok] = new_idx
                self.idx_to_token[new_idx] = tok

    def encode(self, text: str) -> list[int]:
        """Convert text string to sequence of integer token IDs."""
        tokens = tokenize_research_text(text)
        unk_idx = self.token_to_idx[self.unk_token]
        return [self.token_to_idx.get(tok, unk_idx) for tok in tokens]

    def decode(self, indices: list[int]) -> str:
        """Convert list of token IDs back into readable string."""
        return " ".join([self.idx_to_token.get(idx, self.unk_token) for idx in indices])

    def __len__(self):
        return len(self.token_to_idx)

# Example corpus featuring male research collaborators
research_corpus = [
    "Khalid initialized the transformer weights with Xavier normal initialization.",
    "Tariq evaluated the multihead self-attention layers on validation data.",
    "Rahim optimized the AdamW learning rate with a cosine decay schedule.",
    "Karim monitored the cross-entropy loss across 50 epochs.",
    "Farhan logged validation perplexity and saved model checkpoints.",
    "David implemented rotary positional embeddings for long contexts.",
    "Alex analyzed gradient flow through the residual connections."
]

vocab = Vocabulary(min_freq=1)
vocab.build_vocab(research_corpus)

sample_query = "Khalid and Tariq optimized the loss"
encoded_ids = vocab.encode(sample_query)
decoded_text = vocab.decode(encoded_ids)

print(f"Vocabulary Size: {len(vocab)} unique tokens")
print(f"Query: '{sample_query}'")
print(f"Encoded IDs: {encoded_ids}")
print(f"Decoded: '{decoded_text}'")

Vocabulary Size: 59 unique tokens
Query: 'Khalid and Tariq optimized the loss'
Encoded IDs: [5, 40, 12, 21, 2, 33]
Decoded: 'khalid and tariq optimized the loss'


### 🧠 Practice

1. Encode the sentence `"Alex noticed an unknown token"`. Verify that the unknown word maps to the `<unk>` ID (`1`).
2. Write a function `pad_batch(batch_ids, pad_idx=0)` that pads a list of variable-length ID lists to equal length using NumPy.

<details>
<summary>💡 <b>Show solution</b> (try first!)</summary>

```python
test_encoded = vocab.encode("Alex noticed an unknown token")
print("Encoded with UNK:", test_encoded)

import numpy as np
def pad_batch(batch_ids, pad_idx=0):
    max_len = max(len(seq) for seq in batch_ids)
    padded = np.full((len(batch_ids), max_len), fill_value=pad_idx, dtype=int)
    for i, seq in enumerate(batch_ids):
        padded[i, :len(seq)] = seq
    return padded

sample_batch = [[4, 12, 18], [2, 7], [9, 15, 21, 33]]
print("Padded batch array:\n", pad_batch(sample_batch))
```

</details>

## Chapter 29 · Vectorization: BoW and TF-IDF from Scratch ⏱️ ~12 min

Traditional and hybrid ML models require tabular numerical matrices where each row is a document and each column represents a vocabulary feature.

* **Bag of Words (BoW)**: Simply counts how many times word $t$ occurs in document $d$. Disadvantage: frequent non-informative words like `"the"`, `"is"` dominate the gradient.
* **TF-IDF (Term Frequency - Inverse Document Frequency)**: Discounts words that appear everywhere across the dataset:

$$TF(t, d) = 1 + \log(count(t, d)) \quad \text{if } count > 0$$
$$IDF(t, D) = \log\left(\frac{1 + N}{1 + DF(t)}\right) + 1$$

Each row vector is then **$L_2$ normalized** so that long documents don't artificially receive higher dot-product similarity than concise documents.

In [3]:
import numpy as np

class PureNumpyTFIDF:
    """Pure NumPy implementation of TF-IDF feature extraction with L2 normalization."""
    def __init__(self):
        self.vocab = {}
        self.idf_vector = None

    def fit_transform(self, documents: list[str]) -> np.ndarray:
        doc_tokens = [tokenize_research_text(doc) for doc in documents]
        unique_tokens = sorted(list({tok for toks in doc_tokens for tok in toks}))
        self.vocab = {tok: idx for idx, tok in enumerate(unique_tokens)}
        n_docs = len(documents)
        n_vocab = len(self.vocab)
        
        # 1. Raw Count Matrix
        tf_matrix = np.zeros((n_docs, n_vocab), dtype=np.float64)
        for doc_idx, toks in enumerate(doc_tokens):
            for tok in toks:
                if tok in self.vocab:
                    tf_matrix[doc_idx, self.vocab[tok]] += 1.0
                    
        # 2. Sublinear Term Frequency
        tf_scaled = np.zeros_like(tf_matrix)
        pos_mask = tf_matrix > 0
        tf_scaled[pos_mask] = 1.0 + np.log(tf_matrix[pos_mask])
        
        # 3. Document Frequency & Smoothed IDF
        df_vector = np.sum(tf_matrix > 0, axis=0)
        self.idf_vector = np.log((1.0 + n_docs) / (1.0 + df_vector)) + 1.0
        
        # 4. TF-IDF
        tfidf_matrix = tf_scaled * self.idf_vector
        
        # 5. L2 Normalization per document
        norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return tfidf_matrix / norms

    def transform(self, new_docs: list[str]) -> np.ndarray:
        n_docs = len(new_docs)
        n_vocab = len(self.vocab)
        tf_matrix = np.zeros((n_docs, n_vocab), dtype=np.float64)
        
        for doc_idx, doc in enumerate(new_docs):
            for tok in tokenize_research_text(doc):
                if tok in self.vocab:
                    tf_matrix[doc_idx, self.vocab[tok]] += 1.0
                    
        tf_scaled = np.zeros_like(tf_matrix)
        pos_mask = tf_matrix > 0
        tf_scaled[pos_mask] = 1.0 + np.log(tf_matrix[pos_mask])
        
        tfidf_matrix = tf_scaled * self.idf_vector
        norms = np.linalg.norm(tfidf_matrix, axis=1, keepdims=True)
        norms[norms == 0] = 1.0
        return tfidf_matrix / norms

vectorizer = PureNumpyTFIDF()
X_tfidf = vectorizer.fit_transform(research_corpus)
print(f"TF-IDF Matrix Shape: {X_tfidf.shape} (documents x vocabulary)")

# Document Similarity via Cosine Similarity (Dot product of L2-normalized vectors)
sim_0_1 = float(np.dot(X_tfidf[0], X_tfidf[1]))
print(f"Cosine Similarity between Doc 0 (Khalid) and Doc 1 (Tariq): {sim_0_1:.4f}")

TF-IDF Matrix Shape: (7, 57) (documents x vocabulary)
Cosine Similarity between Doc 0 (Khalid) and Doc 1 (Tariq): 0.0344


### 🧠 Practice

1. Given the learned vocabulary, find the word index corresponding to `"transformer"`. What is its IDF score?
2. Write a function `find_most_similar_doc(query, vectorizer, doc_vectors)` that returns the index of the document most similar to `query`.

<details>
<summary>💡 <b>Show solution</b> (try first!)</summary>

```python
idx_transformer = vectorizer.vocab.get("transformer")
idf_val = vectorizer.idf_vector[idx_transformer]
print(f"'transformer' index: {idx_transformer}, IDF: {idf_val:.4f}")

def find_most_similar_doc(query, vec, doc_matrix):
    q_vec = vec.transform([query])[0]
    scores = np.dot(doc_matrix, q_vec)
    best_idx = int(np.argmax(scores))
    return best_idx, scores[best_idx]

best_i, best_sim = find_most_similar_doc("Tariq attention validation", vectorizer, X_tfidf)
print(f"Most similar document index: {best_i} with similarity {best_sim:.4f}")
print("Content:", research_corpus[best_i])
```

</details>

## Chapter 30 · 🏁 NLP Capstone: Peer-Review Classifier from Scratch ⏱️ ~14 min

Let's bring everything from Part D, Part E, and Part F together into a complete machine learning pipeline.

We will build an end-to-end **Peer-Review Sentiment Classifier** in pure Python and NumPy:
1. **Raw review comments** authored by male peer reviewers (Khalid, Tariq, Rahim, Karim, David, Farhan, Alex).
2. **Text tokenization and TF-IDF feature extraction** from scratch.
3. **Logistic Regression training** using the gradient descent algorithm implemented in Chapter 26.
4. **Model inference, accuracy, precision, and confusion matrix evaluation**.

In [4]:
# Peer review dataset labeled by male research reviewers
training_reviews = [
    # Positive peer reviews (1: Accept)
    ("Khalid: Exceptional research contribution with rigorous mathematical proofs.", 1),
    ("Tariq: Solid ablation study and impressive speedup on modern GPUs.", 1),
    ("Rahim: Outstanding empirical evaluation on ImageNet and benchmark datasets.", 1),
    ("Karim: Clear presentation, elegant formulation, and reproducible codebase.", 1),
    ("Farhan: Thorough error analysis and significant improvement over prior baselines.", 1),
    ("David: Novel attention mechanism that scales efficiently to long contexts.", 1),
    ("Alex: Excellent methodological rigor with insightful theoretical guarantees.", 1),
    ("Omar: Comprehensive experimental validation with consistent performance gains.", 1),
    ("Hasan: High quality writing and insightful discussion of ethical limitations.", 1),
    ("Zayn: Compelling results across all five NLP benchmarks with clean code.", 1),
    # Negative peer reviews (0: Reject)
    ("Khalid: Baseline comparisons are missing and empirical claims are unsupported.", 0),
    ("Tariq: Severely flawed experimental methodology and missing hyperparameter details.", 0),
    ("Rahim: Inconsistent notation throughout the paper and confusing proofs.", 0),
    ("Karim: Poor evaluation metrics and trivial performance gain over standard models.", 0),
    ("Farhan: Results cannot be reproduced and code repository contains critical bugs.", 0),
    ("David: Highly marginal contribution with superficial analysis of failures.", 0),
    ("Alex: Lacks novelty and simply combines existing heuristics without insight.", 0),
    ("Omar: Training curves exhibit severe instability and overfitting is ignored.", 0),
    ("Hasan: Unsubstantiated claims with zero statistical significance tests.", 0),
    ("Zayn: The proposed architecture fails to outperform basic linear regression.", 0),
]

review_texts = [r[0] for r in training_reviews]
review_labels = np.array([r[1] for r in training_reviews], dtype=np.float64)

# 1. Feature Extraction
review_vectorizer = PureNumpyTFIDF()
X_train = review_vectorizer.fit_transform(review_texts)
y_train = review_labels

# 2. Pure NumPy Logistic Regression
class NLPLogisticRegression:
    def __init__(self, learning_rate=1.0, epochs=250):
        self.lr = learning_rate
        self.epochs = epochs
        self.weights = None
        self.bias = 0.0
        self.loss_history = []

    def sigmoid(self, z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -250, 250)))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0.0
        
        for _ in range(self.epochs):
            predictions = self.sigmoid(np.dot(X, self.weights) + self.bias)
            
            # Binary Cross Entropy Loss
            eps = 1e-15
            preds_clipped = np.clip(predictions, eps, 1.0 - eps)
            loss = -np.mean(y * np.log(preds_clipped) + (1.0 - y) * np.log(1.0 - preds_clipped))
            self.loss_history.append(loss)
            
            # Gradients
            error = predictions - y
            dw = (1.0 / n_samples) * np.dot(X.T, error)
            db = (1.0 / n_samples) * np.sum(error)
            
            self.weights -= self.lr * dw
            self.bias -= self.lr * db

    def predict_proba(self, X):
        return self.sigmoid(np.dot(X, self.weights) + self.bias)

    def predict(self, X):
        return (self.predict_proba(X) >= 0.5).astype(int)

# Train model
clf = NLPLogisticRegression(learning_rate=1.2, epochs=200)
clf.fit(X_train, y_train)

# Evaluation
train_preds = clf.predict(X_train)
acc = np.mean(train_preds == y_train)
print(f"Final Training BCE Loss: {clf.loss_history[-1]:.4f}")
print(f"Training Accuracy: {acc * 100:.1f}%")

# Inference on unseen reviews
unseen_reviews = [
    "Tariq: Superb experimental evaluation with insightful proofs and clean reproducibility.",
    "Khalid: The experiments lack statistical rigor and baseline results are flawed."
]
X_unseen = review_vectorizer.transform(unseen_reviews)
unseen_probs = clf.predict_proba(X_unseen)

for rev, prob in zip(unseen_reviews, unseen_probs):
    decision = "ACCEPT" if prob >= 0.5 else "REJECT"
    print(f"\nReview: {rev}\nDecision: {decision} (confidence: {prob:.3f})")

Final Training BCE Loss: 0.1112
Training Accuracy: 100.0%

Review: Tariq: Superb experimental evaluation with insightful proofs and clean reproducibility.
Decision: ACCEPT (confidence: 0.771)

Review: Khalid: The experiments lack statistical rigor and baseline results are flawed.
Decision: REJECT (confidence: 0.160)


### 🧠 Practice

1. Identify the top 3 words with the highest positive weights (strongest indicators of paper acceptance).
2. Identify the top 3 words with the lowest negative weights (strongest indicators of rejection).

<details>
<summary>💡 <b>Show solution</b> (try first!)</summary>

```python
vocab_list = sorted(list(review_vectorizer.vocab.items()), key=lambda x: x[1])
idx_to_word = [w for w, i in vocab_list]

top_pos_indices = np.argsort(clf.weights)[-3:][::-1]
top_neg_indices = np.argsort(clf.weights)[:3]

print("Top 3 Acceptance Words:", [idx_to_word[i] for i in top_pos_indices])
print("Top 3 Rejection Words:", [idx_to_word[i] for i in top_neg_indices])
```

</details>